# SASRec Attention-Bias Time-Aware BPI2012 Colab Train (`refine_ml50_do035` baseline)

Colab notebook for the first attention-bias time-aware experiment on top of `refine_ml50_do035`.

Design:
- baseline reuse: `refine_ml50_do035`
- time source: `delta_start_seconds`
- pairwise causal gap attention bias
- 9-bucket scalar attention bias
- evaluate under both `NDCG@10` and `NDCG@5` model-selection criteria


In [ ]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
GITHUB_USERNAME = 'hwbuzz'

DRIVE_ROOT = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction'
REPO_DIR = '/content/time-aware-behavior-prediction'

DATA_DIR = f'{DRIVE_ROOT}/data/processed/bpi2012_complete_only'
BASELINE_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10'
BASELINE_NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5'
TIMEAWARE_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10'
TIMEAWARE_NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg5'
NOTEBOOK_DIR = f'{DRIVE_ROOT}/notebooks'

print('DATA_DIR:', DATA_DIR)
print('BASELINE_NDCG10_OUTPUT_DIR:', BASELINE_NDCG10_OUTPUT_DIR)
print('BASELINE_NDCG5_OUTPUT_DIR:', BASELINE_NDCG5_OUTPUT_DIR)
print('TIMEAWARE_NDCG10_OUTPUT_DIR:', TIMEAWARE_NDCG10_OUTPUT_DIR)
print('TIMEAWARE_NDCG5_OUTPUT_DIR:', TIMEAWARE_NDCG5_OUTPUT_DIR)


In [ ]:
!mkdir -p "$NOTEBOOK_DIR"
!mkdir -p "$DATA_DIR"
!mkdir -p "$BASELINE_NDCG10_OUTPUT_DIR"
!mkdir -p "$BASELINE_NDCG5_OUTPUT_DIR"
!mkdir -p "$TIMEAWARE_NDCG10_OUTPUT_DIR"
!mkdir -p "$TIMEAWARE_NDCG5_OUTPUT_DIR"


In [ ]:
%cd /content
!test -d time-aware-behavior-prediction || git clone https://github.com/$GITHUB_USERNAME/time-aware-behavior-prediction.git
%cd /content/time-aware-behavior-prediction
!git pull


In [ ]:
%cd /content/time-aware-behavior-prediction

skip_packages = ['pywinpty']

with open('requirements.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open('requirements_colab.txt', 'w', encoding='utf-8') as f:
    for line in lines:
        pkg = line.strip().lower()
        if not any(name in pkg for name in skip_packages):
            f.write(line)

print('created requirements_colab.txt')


In [ ]:
!pip install -r requirements_colab.txt


In [ ]:
!ls "$DATA_DIR"


In [ ]:
%cd /content/time-aware-behavior-prediction
!mkdir -p data/processed
!cp -r "$DATA_DIR" data/processed/
!ls data/processed/bpi2012_complete_only


## Experiment design

Fixed baseline setting:
- `refine_ml50_do035`
- `hidden_units=50, num_blocks=2, num_heads=1, maxlen=50, lr=0.001, dropout=0.35`
- seeds: `42`, `2024`, `7`

Time-aware design:
- `delta_start_seconds`
- causal pairwise gap attention bias
- 9-bucket scalar bias
- no additive time embedding in this notebook


## Check existing baseline runs


In [ ]:
from pathlib import Path

baseline_runs = [
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]

for label, output_dir in [
    ('Baseline NDCG@10', Path(BASELINE_NDCG10_OUTPUT_DIR)),
    ('Baseline NDCG@5', Path(BASELINE_NDCG5_OUTPUT_DIR)),
]:
    print('=' * 80)
    print(label)
    for run_name in baseline_runs:
        run_dir = output_dir / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'MISSING')


## Check planned new runs


In [ ]:
planned_ndcg10 = [
    'attnbias_dstart_ml50_do035_b9_s42',
    'attnbias_dstart_ml50_do035_b9_s2024',
    'attnbias_dstart_ml50_do035_b9_s7',
]
planned_ndcg5 = [
    'attnbias_dstart_ml50_do035_b9_s42',
    'attnbias_dstart_ml50_do035_b9_s2024',
    'attnbias_dstart_ml50_do035_b9_s7',
]

for label, output_dir, run_names in [
    ('Attention-Bias NDCG@10', Path(TIMEAWARE_NDCG10_OUTPUT_DIR), planned_ndcg10),
    ('Attention-Bias NDCG@5', Path(TIMEAWARE_NDCG5_OUTPUT_DIR), planned_ndcg5),
]:
    print('=' * 80)
    print(label)
    for run_name in run_names:
        run_dir = output_dir / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'OK')


## Train attention-bias runs for `NDCG@10`


### attnbias_dstart_ml50_do035_b9_s42


In [ ]:
!python src/train_sasrec.py \
  --run_name attnbias_dstart_ml50_do035_b9_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


### attnbias_dstart_ml50_do035_b9_s2024


In [ ]:
!python src/train_sasrec.py \
  --run_name attnbias_dstart_ml50_do035_b9_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


### attnbias_dstart_ml50_do035_b9_s7


In [ ]:
!python src/train_sasrec.py \
  --run_name attnbias_dstart_ml50_do035_b9_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


## Train attention-bias runs for `NDCG@5`


### attnbias_dstart_ml50_do035_b9_s42


In [ ]:
!python src/train_sasrec.py \
  --run_name attnbias_dstart_ml50_do035_b9_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


### attnbias_dstart_ml50_do035_b9_s2024


In [ ]:
!python src/train_sasrec.py \
  --run_name attnbias_dstart_ml50_do035_b9_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


### attnbias_dstart_ml50_do035_b9_s7


In [ ]:
!python src/train_sasrec.py \
  --run_name attnbias_dstart_ml50_do035_b9_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --use_time_attention_bias \
  --time_delta_column delta_start_seconds \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_attention_bias_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


## Rebuild result tables


In [ ]:
from pathlib import Path
import json
import pandas as pd

def rebuild_df(output_dir: str):
    rows = []
    output_path = Path(output_dir)
    if not output_path.exists():
        return pd.DataFrame()
    for run_dir in output_path.iterdir():
        if not run_dir.is_dir():
            continue
        summary_path = run_dir / 'metrics_summary.json'
        config_path = run_dir / 'config.json'
        if not summary_path.exists() or not config_path.exists():
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        config = json.loads(config_path.read_text(encoding='utf-8'))
        row = {
            'run_name': summary.get('run_name'),
            'run_dir': str(run_dir),
            'completed_at': summary.get('completed_at'),
            'best_epoch': summary.get('best_epoch'),
            'checkpoint_best': summary.get('checkpoint_best'),
            'checkpoint_last': summary.get('checkpoint_last'),
            'metrics_history': summary.get('metrics_history'),
            'config_path': str(config_path),
            'metrics_summary': str(summary_path),
            'maxlen': config.get('maxlen'),
            'dropout_rate': config.get('dropout_rate'),
            'hidden_units': config.get('hidden_units'),
            'seed': config.get('seed'),
            'selection_metric': config.get('selection_metric'),
            'use_time_embedding': config.get('use_time_embedding', False),
            'use_time_attention_bias': config.get('use_time_attention_bias', False),
            'time_modeling_mode': config.get('time_modeling_mode'),
            'time_encoding': config.get('time_encoding'),
            'time_delta_column': config.get('time_delta_column'),
            'time_bucket_boundaries_parsed': config.get('time_bucket_boundaries_parsed'),
            'time_attention_bias_bucket_count': config.get('time_attention_bias_bucket_count'),
            'primary_metric_name': config.get('selection_metric'),
        }
        best_valid = summary.get('best_valid', {})
        best_test = summary.get('best_test_at_best_valid', {})
        def pick(metrics_group, mode, key):
            return metrics_group.get(mode, {}).get(key)
        row.update({
            'best_valid_full_ndcg@10': pick(best_valid, 'full', 'ndcg@10'),
            'best_valid_full_hr@10': pick(best_valid, 'full', 'hr@10'),
            'best_valid_full_ndcg@5': pick(best_valid, 'full', 'ndcg@5'),
            'best_valid_full_hr@5': pick(best_valid, 'full', 'hr@5'),
            'best_valid_full_mrr': pick(best_valid, 'full', 'mrr'),
            'best_test_full_ndcg@10': pick(best_test, 'full', 'ndcg@10'),
            'best_test_full_hr@10': pick(best_test, 'full', 'hr@10'),
            'best_test_full_ndcg@5': pick(best_test, 'full', 'ndcg@5'),
            'best_test_full_hr@5': pick(best_test, 'full', 'hr@5'),
            'best_test_full_mrr': pick(best_test, 'full', 'mrr'),
            'best_valid_sampled_ndcg@10': pick(best_valid, 'sampled', 'ndcg@10'),
            'best_valid_sampled_hr@10': pick(best_valid, 'sampled', 'hr@10'),
            'best_valid_sampled_ndcg@5': pick(best_valid, 'sampled', 'ndcg@5'),
            'best_valid_sampled_hr@5': pick(best_valid, 'sampled', 'hr@5'),
            'best_valid_sampled_mrr': pick(best_valid, 'sampled', 'mrr'),
            'best_test_sampled_ndcg@10': pick(best_test, 'sampled', 'ndcg@10'),
            'best_test_sampled_hr@10': pick(best_test, 'sampled', 'hr@10'),
            'best_test_sampled_ndcg@5': pick(best_test, 'sampled', 'ndcg@5'),
            'best_test_sampled_hr@5': pick(best_test, 'sampled', 'hr@5'),
            'best_test_sampled_mrr': pick(best_test, 'sampled', 'mrr'),
        })
        rows.append(row)
    return pd.DataFrame(rows)


## NDCG@10 comparison summary


In [ ]:
baseline_runs = [
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]
timeaware_runs = [
    'attnbias_dstart_ml50_do035_b9_s42',
    'attnbias_dstart_ml50_do035_b9_s2024',
    'attnbias_dstart_ml50_do035_b9_s7',
]

baseline_df = rebuild_df(BASELINE_NDCG10_OUTPUT_DIR)
timeaware_df = rebuild_df(TIMEAWARE_NDCG10_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(baseline_runs)].copy()
baseline_subset['model_variant'] = 'baseline'
baseline_subset['time_variant'] = 'baseline'

timeaware_subset = timeaware_df[timeaware_df['run_name'].isin(timeaware_runs)].copy()
timeaware_subset['model_variant'] = 'timeaware'
timeaware_subset['time_variant'] = 'attention_bias_dstart_b9'

df_ndcg10 = pd.concat([baseline_subset, timeaware_subset], ignore_index=True)
df_ndcg10 = df_ndcg10.sort_values(['time_variant', 'seed', 'run_name']).reset_index(drop=True)
df_ndcg10[[
    'run_name', 'seed', 'time_variant', 'use_time_embedding', 'use_time_attention_bias', 'time_modeling_mode',
    'time_delta_column', 'time_bucket_boundaries_parsed', 'time_attention_bias_bucket_count',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]]


In [ ]:
summary_ndcg10 = df_ndcg10.groupby('time_variant')[[
    'best_valid_full_ndcg@10', 'best_test_full_ndcg@10',
    'best_valid_full_ndcg@5', 'best_test_full_ndcg@5',
    'best_valid_full_mrr', 'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_test_sampled_ndcg@10',
    'best_valid_sampled_ndcg@5', 'best_test_sampled_ndcg@5',
    'best_valid_sampled_mrr', 'best_test_sampled_mrr',
]].agg(['mean', 'std'])
summary_ndcg10


Interpretation guide for NDCG@10:
- compare `best_valid_full_ndcg@10` and `best_test_full_ndcg@10` first
- then check whether sampled and MRR move in the same direction
- baseline is reused; only attention-bias runs are newly trained here


## NDCG@5 comparison summary


In [ ]:
baseline_runs = [
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]
timeaware_runs = [
    'attnbias_dstart_ml50_do035_b9_s42',
    'attnbias_dstart_ml50_do035_b9_s2024',
    'attnbias_dstart_ml50_do035_b9_s7',
]

baseline_df = rebuild_df(BASELINE_NDCG5_OUTPUT_DIR)
timeaware_df = rebuild_df(TIMEAWARE_NDCG5_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(baseline_runs)].copy()
baseline_subset['model_variant'] = 'baseline'
baseline_subset['time_variant'] = 'baseline'

timeaware_subset = timeaware_df[timeaware_df['run_name'].isin(timeaware_runs)].copy()
timeaware_subset['model_variant'] = 'timeaware'
timeaware_subset['time_variant'] = 'attention_bias_dstart_b9'

df_ndcg5 = pd.concat([baseline_subset, timeaware_subset], ignore_index=True)
df_ndcg5 = df_ndcg5.sort_values(['time_variant', 'seed', 'run_name']).reset_index(drop=True)
df_ndcg5[[
    'run_name', 'seed', 'time_variant', 'use_time_embedding', 'use_time_attention_bias', 'time_modeling_mode',
    'time_delta_column', 'time_bucket_boundaries_parsed', 'time_attention_bias_bucket_count',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]]


In [ ]:
summary_ndcg5 = df_ndcg5.groupby('time_variant')[[
    'best_valid_full_ndcg@10', 'best_test_full_ndcg@10',
    'best_valid_full_ndcg@5', 'best_test_full_ndcg@5',
    'best_valid_full_mrr', 'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_test_sampled_ndcg@10',
    'best_valid_sampled_ndcg@5', 'best_test_sampled_ndcg@5',
    'best_valid_sampled_mrr', 'best_test_sampled_mrr',
]].agg(['mean', 'std'])
summary_ndcg5


Interpretation guide for NDCG@5:
- compare `best_valid_full_ndcg@5` and `best_test_full_ndcg@5` first
- then check whether sampled and MRR move in the same direction
- baseline is reused; only attention-bias runs are newly trained here
